In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt
/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/latest.pt
/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/log.jsonl
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_val.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_val.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/test.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_train_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_val_f.bin
/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/latest.pt
/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/log.js

In [2]:
import os
import gc
import json
import time
import math
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tokenizers import Tokenizer
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

# =====================================================================
# 1. Download Phase_6_Summ Dataset (If Not Already Downloaded)
# =====================================================================
TRAIN_RAW_PATH = "/kaggle/working/Phase_6_Summ/Train/train.json"
TEST_RAW_PATH  = "/kaggle/working/Phase_6_Summ/Test/test.json"

if not os.path.exists(TRAIN_RAW_PATH):
    print("Downloading Phase_6_Summ dataset from Hugging Face...")
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    snapshot_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        repo_type="dataset",
        token=HF_TOKEN,
        allow_patterns=["Phase_6_Summ/**"],
        local_dir="/kaggle/working",
    )

# =====================================================================
# 2. Checkpoint & Tokenizer Paths + Wrapper
# =====================================================================
CKPT_PATHS = {
    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",
    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",
}
TOKENIZER_PATH = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"

class Phase1Tokenizer:
    def __init__(self, path=TOKENIZER_PATH):
        self.tok = Tokenizer.from_file(path)
        self.pad_id = 0   # '<pad>'
        self.bos_id = 1   # '<bos>'
        self.eos_id = 2   # '<eos>'
        self.vocab_size = self.tok.get_vocab_size()

    def encode(self, text: str):
        return self.tok.encode(text, add_special_tokens=False).ids

    def decode(self, ids):
        if torch.is_tensor(ids):
            ids = ids.tolist()
        clean = [int(i) for i in ids if int(i) not in (self.pad_id, self.bos_id, self.eos_id) and int(i) >= 0]
        return self.tok.decode(clean, skip_special_tokens=True).strip()

tokenizer = Phase1Tokenizer()

# =====================================================================
# 3. Reconstruct Phase 1 Model Classes (Arch2 Seq2Seq & Arch3 SWA)
# =====================================================================
def apply_rope(x, cos, sin):
    d = x.shape[-1] // 2
    x1, x2 = x[..., :d], x[..., d:]
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, seq_len, device, dtype):
        t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        cos = freqs.cos().to(dtype)[None, None, :, :]
        sin = freqs.sin().to(dtype)[None, None, :, :]
        return cos, sin

# --- ARCH 2 (160.7M Pre-LN Encoder-Decoder Seq2Seq) ---
class Arch2Attention(nn.Module):
    def __init__(self, d_model=768, n_heads=12):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model, bias=True)
        self.k_proj = nn.Linear(d_model, d_model, bias=True)
        self.v_proj = nn.Linear(d_model, d_model, bias=True)
        self.out_proj = nn.Linear(d_model, d_model, bias=True)
        self.rope = RotaryEmbedding(self.head_dim)

    def forward(self, q_in, kv_in, is_causal=False, use_rope=True):
        B, T_q, C = q_in.size()
        T_k = kv_in.size(1)
        q = self.q_proj(q_in).view(B, T_q, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(kv_in).view(B, T_k, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(kv_in).view(B, T_k, self.n_heads, self.head_dim).transpose(1, 2)
        if use_rope:
            cos_q, sin_q = self.rope(T_q, q.device, q.dtype)
            cos_k, sin_k = self.rope(T_k, k.device, k.dtype)
            q = apply_rope(q, cos_q, sin_q)
            k = apply_rope(k, cos_k, sin_k)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T_q, C))

class Arch2FFN(nn.Module):
    def __init__(self, d_model=768, d_ff=3072):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff, bias=True)
        self.linear2 = nn.Linear(d_ff, d_model, bias=True)

    def forward(self, x):
        return self.linear2(F.gelu(self.linear1(x)))

class Arch2EncoderLayer(nn.Module):
    def __init__(self, d_model=768, n_heads=12, d_ff=3072):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn = Arch2Attention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = Arch2FFN(d_model, d_ff)

    def forward(self, x):
        n1 = self.norm1(x)
        x = x + self.self_attn(n1, n1, is_causal=False)
        x = x + self.ffn(self.norm2(x))
        return x

class Arch2DecoderLayer(nn.Module):
    def __init__(self, d_model=768, n_heads=12, d_ff=3072):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn = Arch2Attention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.cross_attn = Arch2Attention(d_model, n_heads)
        self.norm3 = nn.LayerNorm(d_model)
        self.ffn = Arch2FFN(d_model, d_ff)

    def forward(self, x, enc_out):
        n1 = self.norm1(x)
        x = x + self.self_attn(n1, n1, is_causal=True)
        x = x + self.cross_attn(self.norm2(x), enc_out, is_causal=False, use_rope=False)
        x = x + self.ffn(self.norm3(x))
        return x

class Arch2Encoder(nn.Module):
    def __init__(self, n_layers=6, d_model=768, n_heads=12, d_ff=3072, has_final_norm=False):
        super().__init__()
        self.layers = nn.ModuleList([Arch2EncoderLayer(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model) if has_final_norm else None

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.norm(x) if self.norm is not None else x

class Arch2Decoder(nn.Module):
    def __init__(self, n_layers=6, d_model=768, n_heads=12, d_ff=3072, has_final_norm=False):
        super().__init__()
        self.layers = nn.ModuleList([Arch2DecoderLayer(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model) if has_final_norm else None

    def forward(self, x, enc_out):
        for layer in self.layers:
            x = layer(x, enc_out)
        return self.norm(x) if self.norm is not None else x

class Arch2Seq2SeqModel(nn.Module):
    def __init__(self, state_dict=None, vocab_size=40000, d_model=768, n_layers=6, n_heads=12, d_ff=3072):
        super().__init__()
        has_enc_norm = state_dict is not None and "encoder.norm.weight" in state_dict
        has_dec_norm = state_dict is not None and "decoder.norm.weight" in state_dict
        has_final_norm = state_dict is not None and "final_norm.weight" in state_dict
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.encoder = Arch2Encoder(n_layers, d_model, n_heads, d_ff, has_enc_norm)
        self.decoder = Arch2Decoder(n_layers, d_model, n_heads, d_ff, has_dec_norm)
        if has_final_norm:
            self.final_norm = nn.LayerNorm(d_model)
        self.output_projection = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, src_ids, tgt_ids):
        enc_out = self.encoder(self.token_embedding(src_ids))
        dec_out = self.decoder(self.token_embedding(tgt_ids), enc_out)
        if hasattr(self, "final_norm"):
            dec_out = self.final_norm(dec_out)
        return self.output_projection(dec_out)

# --- ARCH 3 (283.5M 16-Layer SWA Decoder) ---
class Arch3MLP(nn.Module):
    def __init__(self, d_model=1024, d_ff=4096):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff, bias=True)
        self.fc2 = nn.Linear(d_ff, d_model, bias=True)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

class Arch3SWAAttention(nn.Module):
    def __init__(self, d_model=1024, n_heads=16, window_size=256):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.window_size = window_size
        self.q_proj = nn.Linear(d_model, d_model, bias=True)
        self.k_proj = nn.Linear(d_model, d_model, bias=True)
        self.v_proj = nn.Linear(d_model, d_model, bias=True)
        self.out_proj = nn.Linear(d_model, d_model, bias=True)
        self.rope = RotaryEmbedding(self.head_dim)

    def forward(self, x):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        cos, sin = self.rope(T, x.device, x.dtype)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, C))

class Arch3Block(nn.Module):
    def __init__(self, d_model=1024, n_heads=16, d_ff=4096, window_size=256):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = Arch3SWAAttention(d_model, n_heads, window_size)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = Arch3MLP(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class Arch3SWAModel(nn.Module):
    def __init__(self, vocab_size=40000, d_model=1024, n_layers=16, n_heads=16, d_ff=4096, window_size=256):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([Arch3Block(d_model, n_heads, d_ff, window_size) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids, attention_mask=None):
        x = self.token_emb(input_ids)
        for blk in self.blocks:
            x = blk(x)
        return self.lm_head(self.final_norm(x))

def load_phase1_model(arch_name: str, device="cuda"):
    ckpt = torch.load(CKPT_PATHS[arch_name], map_location="cpu", weights_only=False)
    raw_sd = ckpt["model"]
    sd = {k.replace("_orig_mod.", ""): v for k, v in raw_sd.items()}
    if arch_name == "arch2":
        model = Arch2Seq2SeqModel(state_dict=sd)
    elif arch_name == "arch3":
        model = Arch3SWAModel()
    else:
        raise ValueError(f"Unknown arch: {arch_name}")
    missing, unexpected = model.load_state_dict(sd, strict=False)
    print(f"[{arch_name.upper()}] Loaded! Missing: {missing} | Unexpected: {unexpected}")
    del ckpt, raw_sd, sd
    return model.to(device)

# =====================================================================
# 4. Load Splits & Pre-Tokenize Once (~20s)
# =====================================================================
with open(TRAIN_RAW_PATH, "r", encoding="utf-8") as f:
    raw_train_all = json.load(f)
with open(TEST_RAW_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

doc_words = [len(x["document"].split()) for x in raw_train_all]
bins = np.digitize(doc_words, bins=np.quantile(doc_words, [0.25, 0.5, 0.75]))
train_split, val_split = train_test_split(raw_train_all, test_size=0.05, random_state=42, stratify=bins)

train_clean = [
    x for x in train_split
    if len(x["summary"].strip().split()) >= 10 and len(x["document"].strip().split()) >= 20
]

print(f"Pre-tokenizing {len(train_clean)} train & {len(test_data)} test docs...")
t0 = time.time()
for ex in train_clean:
    ex["doc_ids"] = tokenizer.encode(ex["document"])
    ex["sum_ids"] = tokenizer.encode(ex["summary"])
for ex in test_data:
    ex["doc_ids"] = tokenizer.encode(ex["document"])
print(f"Pre-tokenization finished in {time.time() - t0:.1f}s!")

# =====================================================================
# 5. PEFT Modules (LoRA & Pfeiffer Adapters) & Long-Doc Condensers
# =====================================================================
class LoRALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, rank: int = 32, alpha: float = 64.0, dropout: float = 0.05):
        super().__init__()
        self.base_layer = base_layer
        for p in self.base_layer.parameters():
            p.requires_grad = False
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout)
        self.lora_A = nn.Parameter(torch.empty(rank, base_layer.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base_layer.out_features, rank))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x):
        return self.base_layer(x) + (self.dropout(x) @ self.lora_A.T @ self.lora_B.T) * self.scaling

class PfeifferAdapterLinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, bottleneck_dim: int = 96, dropout: float = 0.05):
        super().__init__()
        self.base_layer = base_layer
        for p in self.base_layer.parameters():
            p.requires_grad = False
        d = base_layer.out_features
        self.ln = nn.LayerNorm(d)
        self.down = nn.Linear(d, bottleneck_dim)
        self.act = nn.SiLU()
        self.up = nn.Linear(bottleneck_dim, d)
        self.drop = nn.Dropout(dropout)
        nn.init.normal_(self.down.weight, std=0.01)
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        h = self.base_layer(x)
        return h + self.up(self.act(self.down(self.drop(self.ln(h)))))

def inject_advanced_peft(model: nn.Module, method: str = "lora", rank: int = 32):
    for p in model.parameters():
        p.requires_grad = False

    skip_names = ("token_emb", "token_embedding", "pos_emb", "lm_head", "output_projection")
    for name, module in list(model.named_modules()):
        if isinstance(module, nn.Linear) and not any(s in name for s in skip_names):
            parent_name, child_name = name.rsplit(".", 1) if "." in name else ("", name)
            parent = model.get_submodule(parent_name) if parent_name else model

            if method == "lora":
                setattr(parent, child_name, LoRALinear(module, rank=rank, alpha=rank * 2.0))
            elif method == "pfeiffer":
                if any(t in name for t in ("fc2", "linear2")):
                    setattr(parent, child_name, PfeifferAdapterLinear(module, bottleneck_dim=rank * 3))
                else:
                    setattr(parent, child_name, LoRALinear(module, rank=rank, alpha=rank * 2.0))

    for m in model.modules():
        if isinstance(m, nn.LayerNorm):
            for p in m.parameters():
                p.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Applied {method.upper()}: {trainable:,} trainable / {total:,} total ({100*trainable/total:.2f}%)")
    return model

def condense_ids_v2(all_ids: list, max_tokens: int, strategy: str):
    if len(all_ids) <= max_tokens:
        return all_ids
    if strategy == "section_aware_5part":
        h_len = int(max_tokens * 0.25)
        t_len = int(max_tokens * 0.45)
        rem = max_tokens - h_len - t_len
        c_len = rem // 3
        mid_pool = all_ids[h_len:-t_len]
        if len(mid_pool) <= rem:
            return (all_ids[:h_len] + mid_pool + all_ids[-t_len:])[:max_tokens]
        s1 = int(len(mid_pool) * 0.20)
        s2 = int(len(mid_pool) * 0.50)
        s3 = int(len(mid_pool) * 0.80)
        mid_chunks = (
            mid_pool[s1:s1 + c_len] +
            mid_pool[s2:s2 + c_len] +
            mid_pool[s3:s3 + (rem - 2 * c_len)]
        )
        return (all_ids[:h_len] + mid_chunks + all_ids[-t_len:])[:max_tokens]
    else:  # 'overlapping_head_tail'
        h_len = int(max_tokens * 0.35)
        t_len = int(max_tokens * 0.45)
        m_len = max_tokens - h_len - t_len
        late_mid_start = max(h_len, len(all_ids) - t_len - m_len - int(len(all_ids) * 0.05))
        return (all_ids[:h_len] + all_ids[late_mid_start:late_mid_start + m_len] + all_ids[-t_len:])[:max_tokens]

class Phase6DatasetV2(Dataset):
    def __init__(self, examples, tok, is_seq2seq=False, max_src_len=832, max_tgt_len=384, strategy="section_aware_5part"):
        self.examples = examples
        self.tok = tok
        self.is_seq2seq = is_seq2seq
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len
        self.strategy = strategy
        self.sep_ids = self.tok.encode("\n\nSummary:\n")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        src_ids = condense_ids_v2(ex["doc_ids"], self.max_src_len, self.strategy)
        sum_ids = ex["sum_ids"][:self.max_tgt_len - 1] + [self.tok.eos_id]
        if self.is_seq2seq:
            tgt_in = [self.tok.bos_id] + sum_ids[:-1]
            return {
                "src_ids": torch.tensor(src_ids, dtype=torch.long),
                "tgt_ids": torch.tensor(tgt_in, dtype=torch.long),
                "labels": torch.tensor(sum_ids, dtype=torch.long),
            }
        else:
            prompt_ids = [self.tok.bos_id] + src_ids + self.sep_ids
            full_ids = prompt_ids + sum_ids
            labels = ([-100] * len(prompt_ids)) + sum_ids
            return {
                "input_ids": torch.tensor(full_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
            }

def collate_fn_causal(batch):
    input_ids = nn.utils.rnn.pad_sequence([b["input_ids"] for b in batch], batch_first=True, padding_value=0)
    labels = nn.utils.rnn.pad_sequence([b["labels"] for b in batch], batch_first=True, padding_value=-100)
    return {"input_ids": input_ids, "labels": labels}

def collate_fn_seq2seq(batch):
    src_ids = nn.utils.rnn.pad_sequence([b["src_ids"] for b in batch], batch_first=True, padding_value=0)
    tgt_ids = nn.utils.rnn.pad_sequence([b["tgt_ids"] for b in batch], batch_first=True, padding_value=0)
    labels = nn.utils.rnn.pad_sequence([b["labels"] for b in batch], batch_first=True, padding_value=-100)
    return {"src_ids": src_ids, "tgt_ids": tgt_ids, "labels": labels}

# =====================================================================
# 6. Codabench Root Packaging & Generation
# =====================================================================
def package_codabench_submission(source_json_path: str, output_zip_path: str):
    with open(TEST_RAW_PATH, "r", encoding="utf-8") as f:
        raw_test = json.load(f)
    expected_ids = [str(item["id"]) for item in raw_test]

    with open(source_json_path, "r", encoding="utf-8") as f:
        preds = json.load(f)
    pred_map = {str(item["id"]): str(item["summary"]).strip() for item in preds}

    clean_predictions = []
    for doc_id, test_item in zip(expected_ids, raw_test):
        summary_text = pred_map.get(doc_id, "").strip()
        if not summary_text:
            summary_text = test_item["document"][:600].strip()
        clean_predictions.append({"id": doc_id, "summary": summary_text})

    tmp_pred_path = "/kaggle/working/prediction.json"
    with open(tmp_pred_path, "w", encoding="utf-8") as f:
        json.dump(clean_predictions, f, indent=2, ensure_ascii=False)

    with zipfile.ZipFile(output_zip_path, mode="w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(tmp_pred_path, arcname="prediction.json")

    with zipfile.ZipFile(output_zip_path, mode="r") as zf:
        assert zf.namelist() == ["prediction.json"], f"Bad zip structure: {zf.namelist()}"

    print(f"✅ Packaged Codabench Archive: {output_zip_path} ({os.path.getsize(output_zip_path)/1024:.1f} KB) -> ['prediction.json']")

@torch.no_grad()
def generate_test_summaries_v2(model, examples, is_seq2seq=False, max_src_len=832, max_new_tokens=340, strategy="section_aware_5part", device="cuda"):
    model.eval()
    results = []
    sep_ids = tokenizer.encode("\n\nSummary:\n")
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    for idx, ex in enumerate(examples):
        raw_doc_ids = ex.get("doc_ids") or tokenizer.encode(ex["document"])
        src_ids = condense_ids_v2(raw_doc_ids, max_src_len, strategy)
        generated = []

        if is_seq2seq:
            src_tensor = torch.tensor([src_ids], dtype=torch.long, device=device)
            with torch.amp.autocast(device_type="cuda", dtype=amp_dtype):
                enc_out = model.encoder(model.token_embedding(src_tensor))
            tgt_ids = [tokenizer.bos_id]
            for _ in range(max_new_tokens):
                tgt_tensor = torch.tensor([tgt_ids], dtype=torch.long, device=device)
                with torch.amp.autocast(device_type="cuda", dtype=amp_dtype):
                    dec_out = model.decoder(model.token_embedding(tgt_tensor), enc_out)
                    if hasattr(model, "final_norm"):
                        dec_out = model.final_norm(dec_out)
                    next_logits = model.output_projection(dec_out[:, -1, :]).squeeze(0).float()

                for prev_id in set(generated[-64:]):
                    next_logits[prev_id] = next_logits[prev_id] / 1.2 if next_logits[prev_id] > 0 else next_logits[prev_id] * 1.2
                if len(generated) >= 2:
                    p2 = (generated[-2], generated[-1])
                    for i in range(len(generated) - 2):
                        if (generated[i], generated[i+1]) == p2:
                            next_logits[generated[i+2]] = -1e9

                next_id = int(torch.argmax(next_logits).item())
                if next_id == tokenizer.eos_id and len(generated) >= 50:
                    break
                generated.append(next_id)
                tgt_ids.append(next_id)
        else:
            input_ids = [tokenizer.bos_id] + src_ids + sep_ids
            for _ in range(max_new_tokens):
                inp_tensor = torch.tensor([input_ids], dtype=torch.long, device=device)
                with torch.amp.autocast(device_type="cuda", dtype=amp_dtype):
                    next_logits = model(inp_tensor)[0, -1, :].float()

                for prev_id in set(generated[-64:]):
                    next_logits[prev_id] = next_logits[prev_id] / 1.2 if next_logits[prev_id] > 0 else next_logits[prev_id] * 1.2
                if len(generated) >= 2:
                    p2 = (generated[-2], generated[-1])
                    for i in range(len(generated) - 2):
                        if (generated[i], generated[i+1]) == p2:
                            next_logits[generated[i+2]] = -1e9

                next_id = int(torch.argmax(next_logits).item())
                if next_id == tokenizer.eos_id and len(generated) >= 50:
                    break
                generated.append(next_id)
                input_ids.append(next_id)

        summary_text = tokenizer.decode(generated)
        if not summary_text:
            summary_text = ex["document"][:600].strip()
        results.append({"id": str(ex["id"]), "summary": summary_text})

        if (idx + 1) % 25 == 0:
            print(f"    Generated {idx + 1}/{len(examples)} summaries...")

    return results

def run_additional_experiment(
    exp_name: str,
    arch_name: str,
    peft_method: str,
    long_doc_strategy: str,
    rank: int = 32,
    max_steps: int = 1800,
    lr: float = 4e-4,
    batch_size: int = 2,
    grad_accum: int = 8,
    max_src_len: int = 832,
    max_tgt_len: int = 384,
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    is_seq2seq = (arch_name == "arch2")
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    print(f"\n=======================================================")
    print(f"🚀 Running {exp_name} | Arch: {arch_name.upper()} | Method: {peft_method.upper()} | Strategy: {long_doc_strategy}")
    print(f"=======================================================")

    model = load_phase1_model(arch_name, device=device)
    model = inject_advanced_peft(model, method=peft_method, rank=rank).to(device)

    train_ds = Phase6DatasetV2(train_clean, tokenizer, is_seq2seq=is_seq2seq, max_src_len=max_src_len, max_tgt_len=max_tgt_len, strategy=long_doc_strategy)
    collate = collate_fn_seq2seq if is_seq2seq else collate_fn_causal
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate)

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
    total_opt_steps = max_steps // grad_accum
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_opt_steps), eta_min=lr * 0.1)
    scaler = torch.cuda.amp.GradScaler(enabled=(amp_dtype == torch.float16))

    model.train()
    t0 = time.time()
    global_step = 0
    running_loss = 0.0
    optimizer.zero_grad()

    while global_step < max_steps:
        for batch in train_loader:
            global_step += 1
            with torch.amp.autocast(device_type="cuda", dtype=amp_dtype):
                if is_seq2seq:
                    src_ids = batch["src_ids"].to(device)
                    tgt_ids = batch["tgt_ids"].to(device)
                    labels = batch["labels"].to(device)
                    logits = model(src_ids, tgt_ids)
                    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100)
                else:
                    input_ids = batch["input_ids"].to(device)
                    labels = batch["labels"].to(device)
                    logits = model(input_ids)
                    shift_logits = logits[:, :-1, :].contiguous()
                    shift_labels = labels[:, 1:].contiguous()
                    loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), ignore_index=-100)

                loss = loss / grad_accum

            scaler.scale(loss).backward()
            running_loss += loss.item() * grad_accum

            if global_step % grad_accum == 0 or global_step == max_steps:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

            if global_step % 300 == 0:
                print(f"  Step {global_step}/{max_steps} | Avg Loss: {running_loss / global_step:.4f} | Elapsed: {(time.time()-t0)/60:.1f}m")

            if global_step >= max_steps:
                break

    os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
    torch.save(model.state_dict(), f"/kaggle/working/checkpoints/{exp_name}.pt")

    print(f"Generating 100 test summaries for {exp_name}...")
    submissions = generate_test_summaries_v2(
        model, test_data, is_seq2seq=is_seq2seq, max_src_len=max_src_len, max_new_tokens=340, strategy=long_doc_strategy, device=device
    )

    json_path = f"/kaggle/working/submission_{exp_name}.json"
    zip_path  = f"/kaggle/working/submission_{exp_name}.zip"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(submissions, f, indent=2, ensure_ascii=False)

    package_codabench_submission(json_path, zip_path)
    del model
    gc.collect()
    torch.cuda.empty_cache()

# =====================================================================
# 7. Execute Both New Experiments
# =====================================================================
# Submission 5: Arch3 (283.5M SWA) + LoRA (rank=32) + 5-Part Section-Aware Condenser
run_additional_experiment(
    exp_name="exp5_arch3_lora",
    arch_name="arch3",
    peft_method="lora",
    long_doc_strategy="section_aware_5part",
    rank=32,
    max_steps=1800,
    lr=4e-4
)

# Submission 6: Arch2 (160.7M Seq2Seq) + Pfeiffer Adapters & Cross-Attn LoRA + Overlapping Head-Tail
run_additional_experiment(
    exp_name="exp6_arch2_pfeiffer",
    arch_name="arch2",
    peft_method="pfeiffer",
    long_doc_strategy="overlapping_head_tail",
    rank=32,
    max_steps=1800,
    lr=5e-4
)

print("\n🎉 Done! Download both files from /kaggle/working/:")
print("  1. submission_exp5_arch3_lora.zip")
print("  2. submission_exp6_arch2_pfeiffer.zip")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Pre-tokenizing 6677 train & 100 test docs...
Pre-tokenization finished in 89.5s!

🚀 Running exp5_arch3_lora | Arch: ARCH3 | Method: LORA | Strategy: section_aware_5part
[ARCH3] Loaded! Missing: [] | Unexpected: []
Applied LORA: 9,504,768 trainable / 292,898,816 total (3.25%)


/tmp/ipykernel_24/2012420875.py:561: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(amp_dtype == torch.float16))


  Step 300/1800 | Avg Loss: 6.2430 | Elapsed: 8.4m
  Step 600/1800 | Avg Loss: 5.6513 | Elapsed: 16.9m
  Step 900/1800 | Avg Loss: 5.2792 | Elapsed: 25.4m
  Step 1200/1800 | Avg Loss: 5.0037 | Elapsed: 33.9m
  Step 1500/1800 | Avg Loss: 4.8074 | Elapsed: 42.4m
  Step 1800/1800 | Avg Loss: 4.6576 | Elapsed: 50.8m
Generating 100 test summaries for exp5_arch3_lora...
    Generated 25/100 summaries...
    Generated 50/100 summaries...
    Generated 75/100 summaries...
    Generated 100/100 summaries...
✅ Packaged Codabench Archive: /kaggle/working/submission_exp5_arch3_lora.zip (37.9 KB) -> ['prediction.json']

🚀 Running exp6_arch2_pfeiffer | Arch: ARCH2 | Method: PFEIFFER | Strategy: overlapping_head_tail
[ARCH2] Loaded! Missing: [] | Unexpected: []
Applied PFEIFFER: 6,857,856 trainable / 167,489,664 total (4.09%)
  Step 300/1800 | Avg Loss: 5.9083 | Elapsed: 2.1m
  Step 600/1800 | Avg Loss: 5.2062 | Elapsed: 4.3m
  Step 900/1800 | Avg Loss: 4.8275 | Elapsed: 6.4m
  Step 1200/1800 | Avg L